<a href="https://colab.research.google.com/github/syedebrah/CodeBase/blob/main/SLM_MODEL(QWEN_MODEL).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# Prevent PyTorch memory fragmentation just to be safe
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
# 1. Install Unsloth and Dependencies (Notice 'trl' is no longer restricted!)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes datasets wandb
!pip install -U trl

In [5]:
  # 2. Authenticate Hugging Face
from huggingface_hub import login
print("Login to Hugging Face:")
login() # Paste your write-access token here

Login to Hugging Face:


In [6]:
# 3. Authenticate Weights & Biases
import wandb
print("\nLogin to Weights & Biases:")
wandb.login() # Paste your API key here


Login to Weights & Biases:


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hellomcc123 (hellomcc123-madras-christian-college-chennai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.3 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [8]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 1. Standardize formatting to ChatML
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
    map_eos_token = True,
)

# 2. Load and filter the data
dataset = load_dataset("nampdn-ai/tiny-codes", split="train")
js_dataset = dataset.filter(lambda row: row["programming_language"].lower() == "javascript")

# 3. Apply the conversational template
def format_dataset(examples):
    queries = examples["prompt"]
    codes = examples["response"]

    formatted_texts = []
    for query, code in zip(queries, codes):
        conversation = [
            {"role": "user", "content": query},
            {"role": "assistant", "content": f"```javascript\n{code}\n```"}
        ]
        text = tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

js_dataset = js_dataset.map(format_dataset, batched=True)

Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


README.md: 0.00B [00:00, ?B/s]

part_1_200000.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

part_2_400000.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

part_3_600000.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

...

part_8_1600000.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

part_9_1632520.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1632309 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1632309 [00:00<?, ? examples/s]

Map:   0%|          | 0/131014 [00:00<?, ? examples/s]

In [9]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import os
import wandb

# 1. Initialize the JSSLM tracking project in WandB
run_name = "qwen-run1"
wandb.init(project="Qwen", name=run_name)
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# 2. Configure the Trainer
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = js_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",
        run_name = run_name,

        dataset_text_field = "text",
        max_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
    ),
)

# 3. Start Training!
trainer_stats = trainer.train()

# 4. Close WandB gracefully
wandb.finish()

wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/131014 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 131,014 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,1.808236
2,1.879275
3,1.783039
4,1.895929
5,1.796865
6,1.666158
7,1.633740
8,1.566877
9,1.626171
10,1.405768


wandb: Adding directory to artifact (outputs/checkpoint-60)... Done. 0.3s


train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▆▅▅▄▃▃▃▃▄▆█▆█▄▅▄▄▄▂▃▃▃▃▂▃▃▃▃▂▂▁▁▁▁▁▁▂▁▂▁
train/learning_rate,▁▂▄▅▇██▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁
train/loss,▇█▆▆▆▅▅▅▃▅▃▃▃▃▂▂▁▂▂▃▂▂▂▂▂▁▁▁▂▂▂▂▁▂▂▂▂▁▁▁
total_flos,929553169420800.0
train/epoch,0.00733
train/global_step,60
train/grad_norm,0.28891
train/learning_rate,0.0
train/loss,0.88061


In [10]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a JavaScript Code to allow voters based on the Age ."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure JavaScript ---


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

```javascript
Here's some sample code in Javascript that allows voters based on their age:

 ```javascript
function checkAge(age) {
  if (age >= 18 && age <= 65) {
    console.log("You are eligible for voting!");
  } else if (age < 18) {
    console.log("Sorry, you must be at least 18 years old to vote.");
  } else {
    console.log("Invalid input. Please enter a valid number.");
  }
}
```

In this code, we define a function called `checkAge` which takes one parameter called `age`. The function checks whether the age is between 18 and 65 inclusive. If so, it logs "You are eligible for voting!" to the console. Otherwise, it checks whether the age is less than 18 but greater than or equal to 18. If so, it logs "Sorry, you must be at least 18 years old to vote." to the console. Finally, if neither of these conditions are met, it logs "Invalid input. Please enter a valid number." to the console.
```<|im_end|>


In [11]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a JavaScript snippet that selects a button with the ID 'theme-toggle' and adds a click event listener that toggles a dark-mode class on the document body."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure JavaScript ---
```javascript
// Select the button element by its ID
const themeToggle = document.getElementById('theme-toggle');

// Add a click event listener to the button
themeToggle.addEventListener('click', function() {
  // Toggle the 'dark-mode' class on the document body
  document.body.classList.toggle('dark-mode');
});
```
This code first selects the button element using its ID. Then, it attaches a click event listener to this element. When the button is clicked, the `toggle` method of the `classList` property is called on the document body. This method checks whether the 'dark-mode' class exists as a child of the body element. If it does not exist, it adds it; otherwise, it removes it. This allows for easy toggling between light and dark modes without needing to refresh the page.<|im_end|>


In [12]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write an asynchronous JavaScript function that fetches data from a 'students.json' file. Once the JSON is parsed, use the array filter method to return only the records where the 'department' property is exactly 'BCA' and the 'year' property is exactly 1."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure JavaScript ---
```javascript
// Import necessary package for reading JSON files
const fs = require('fs');

/**
 * Fetches data from a JSON file and returns filtered results.
 * 
 * @param {string} filePath - The path to the JSON file.
 * @returns {Array} An array of objects containing filtered data.
 */
async function fetchData(filePath) {
  try {
    // Read JSON file asynchronously
    const jsonData = await fs.promises.readFile(filePath, 'utf8');
    
    // Parse JSON data into JavaScript object
    const studentsData = JSON.parse(jsonData);
    
    // Filter students based on specific criteria
    const filteredStudents = studentsData.filter(student => {
      return student.department === 'BCA' && student.year === 1;
    });
    
    return filteredStudents;
  } catch (error) {
    console.error(`Error fetching data: ${error.message}`);
    throw error;
  }
}

// Example usage
(async () => {
  try {
    const filteredStudents = await fetchData('students.json'

In [13]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a frontend JavaScript function using the browser's native Fetch API to retrieve data from the 'students.json' endpoint. Once the JSON is parsed, use the array filter method to return only the records where the 'department' property is exactly 'BCA' and the 'year' property is exactly 1. Include a try/catch block for error handling."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure JavaScript ---
```javascript
// Import necessary package
const fetch = require('node-fetch');

/**
 * Retrieves students from the 'students.json' endpoint where department is 'BCA' and year is 1.
 * @returns {Promise<Array>} A promise containing an array of student objects.
 */
async function getStudents() {
  try {
    // Make a GET request to the 'students.json' endpoint
    const response = await fetch('https://jsonplaceholder.typicode.com/users');
    
    // Parse the JSON response
    const data = await response.json();
    
    // Filter the data based on the specified conditions
    const filteredData = data.filter(student => 
      student.department === 'BCA' && student.year === 1);
    
    return filteredData;
  } catch (error) {
    console.error('Error fetching data:', error);
    throw new Error('Failed to fetch data');
  }
}

// Example usage
getStudents().then(students => {
  console.log(students);
}).catch(error => {
  console.error('Error retrievi

In [15]:
messages = [
    {"role": "user", "content": "Write a frontend JavaScript function using the browser's native Fetch API to retrieve 'students.json'. Filter and return only the records where department is 'BCA' and year is 1. Do not use require or imports. Start the code immediately."},
]

# 1. Apply the template but keep it as a string
prompt_string = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

# 2. THE TRICK: Force the model to start inside the function!
prompt_string += "```javascript\nasync function getBCAStudents() {\n"

# 3. Tokenize and send to GPU
inputs = tokenizer([prompt_string], return_tensors = "pt").to("cuda")

print("--- Generating Pure Browser JavaScript ---")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure Browser JavaScript ---
  const response = await fetch('students.json');
  const data = await response.json();
  
  return data.filter(student => student.department === 'BCA' && student.year === 1);
}
```<|im_end|>


In [16]:
from transformers import TextStreamer

# Enable 2x faster native inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Create a JS code to Retreive all the details from the Student database from MYSQL with table name MSC data Science"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("--- Generating Pure JavaScript ---")
_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.1,
)

--- Generating Pure JavaScript ---
Here is some sample Javascript code which connects to a MySQL database and retrieves all the details from the `student` table:

```javascript
const mysql = require('mysql');

// Create a connection object
const conn = mysql.createConnection({
  host: 'localhost',
  user: 'root',
  password: '',
  database: 'msc'
});

// Connect to the database
conn.connect((err) => {
  if (err) throw err;
  console.log('Connected!');
  
  // Query the database
  conn.query('SELECT * FROM student', (err, results) => {
    if (err) throw err;
    
    // Print the results
    console.log(results);
  });
});
```

This code uses the `mysql` module to create a connection object to the MySQL database. It then connects to the database using the `connect()` method and passes in the necessary parameters such as the host, username, password, and database name. Once connected, the code queries the `student` table using the `query()` method and passes in the SQL query string. The

In [ ]:
ss